#### Recovery: Manages failures and exceptions gracefully in agent workflows. This component implements retry logic, fallback processes, and error handling to ensure system resilience.


#### NOTE: i am using github openai for this example.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from pydantic import BaseModel
from typing import Optional

In [2]:
TOKEN = os.getenv('GITHUB_TOKEN')
ENDPOINT = os.getenv('GITHUB_ENDPOINT')
MODEL = os.getenv('GITHUB_MODEL_NAME')

client = OpenAI(
    base_url=ENDPOINT,
    api_key=TOKEN,
)

In [18]:
class UserInfo(BaseModel):
    name: str
    email: str 
    age: Optional[int] = None

def resilient_intelligence(prompt: str) -> str:
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "Extract user information from the text. Reply ONLY with a valid JSON object: {\"name\": str, \"email\": str, \"age\": int (optional)}"},
            {"role": "user", "content": prompt},
        ],
        model=MODEL,
        response_format={"type": "json_object"},
    )

    print(f"🔍 Response: {response.choices[0].message.content}")

    user_data = UserInfo.model_validate_json(response.choices[0].message.content)

    try:
        age = user_data.age
        if age is None:
            raise ValueError("Age is None!!!")
        return f"Name: {user_data.name}, Email: {user_data.email}, Age: {age}"
    
    except (KeyError, TypeError, ValueError):
        print("❌ age not found, using fallback info...")
        return f"Name: {user_data.name}, Email: {user_data.email}, Age: Not provided"
    

In [20]:
res = resilient_intelligence(
    prompt="my name is jiten and my email is j@gmail.com"
)
print('✅recovery output: ', res)

🔍 Response: {"name": "jiten", "email": "j@gmail.com"}
❌ age not found, using fallback info...
✅recovery output:  Name: jiten, Email: j@gmail.com, Age: Not provided
